This notebook handles:
1. Loading and cleaning the Superstore dataset
2. Connecting to SQL Server
3. Inserting data into 4 normalized tables

## Step 1 — Load & Explore the Dataset

In [1]:
import pandas as pd

# Load raw CSV
df = pd.read_csv(
    r'C:\Users\HP\Desktop\Celebal_Internship_Assignment\Superstore_Dataset\archive\Sample - Superstore.csv',
    encoding='latin1'
)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (9994, 21)
Columns: ['ï»¿Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,ï»¿Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,06-12-2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 2 — Clean & Prepare the Dataset

In [2]:
# Save as pipe-separated file to avoid comma conflicts in product names
df.to_csv(
    r'C:\Users\HP\Desktop\Celebal_Internship_Assignment\Superstore_Dataset\archive\Superstore_clean.csv',
    index=False,
    sep='|',
    encoding='utf-8'
)

# Reload clean file and fix column names
df = pd.read_csv(
    r'C:\Users\HP\Desktop\Celebal_Internship_Assignment\Superstore_Dataset\archive\Superstore_clean.csv',
    sep='|',
    encoding='utf-8-sig'
)

# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
df.rename(columns={df.columns[0]: 'Row_ID'}, inplace=True)

print(f"Columns after cleaning: {df.columns.tolist()}")
print(f"Total rows: {len(df)}")
df.head()

Columns after cleaning: ['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Total rows: 9994


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,06-12-2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 3 — Connect to SQL Server

In [3]:
import pyodbc
from sqlalchemy import create_engine

# Verify connection
conn_str = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=superstore;Trusted_Connection=yes;'
try:
    conn = pyodbc.connect(conn_str)
    print('Connected to SQL Server successfully!')
except Exception as e:
    print(f'Connection failed: {e}')

# Create SQLAlchemy engine
engine = create_engine('mssql+pyodbc://localhost/superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes')
print('Engine created!')

Connected to SQL Server successfully!
Engine created!


## Step 4 — Insert Data into 4 Normalized Tables

The Superstore dataset is split into 4 tables:
- `customers` — unique customer information
- `products` — unique product information  
- `orders` — order-level data
- `order_items` — individual items per order

In [4]:
# 1. Customers table
customers = df[['Customer_ID','Customer_Name','Segment','Country','City','State','Postal_Code','Region']].drop_duplicates(subset='Customer_ID')
customers.columns = ['customer_id','customer_name','segment','country','city','state','postal_code','region']
customers.to_sql('customers', con=engine, if_exists='replace', index=False)
print(f'Customers inserted: {len(customers)} rows')

# 2. Products table
products = df[['Product_ID','Category','Sub_Category','Product_Name','Sales']].drop_duplicates(subset='Product_ID')
products.columns = ['product_id','category','sub_category','product_name','unit_price']
products.to_sql('products', con=engine, if_exists='replace', index=False)
print(f'Products inserted: {len(products)} rows')

# 3. Orders table
orders = df[['Order_ID','Customer_ID','Order_Date','Ship_Date','Ship_Mode','Sales']].drop_duplicates(subset='Order_ID')
orders.columns = ['order_id','customer_id','order_date','ship_date','ship_mode','total_amount']
orders['status'] = 'Delivered'
orders.to_sql('orders', con=engine, if_exists='replace', index=False)
print(f'Orders inserted: {len(orders)} rows')

# 4. Order Items table
order_items = df[['Row_ID','Order_ID','Product_ID','Quantity','Sales','Discount']]
order_items.columns = ['row_id','order_id','product_id','quantity','unit_price','discount_pct']
order_items.to_sql('order_items', con=engine, if_exists='replace', index=False)
print(f'Order Items inserted: {len(order_items)} rows')

Customers inserted: 793 rows
Products inserted: 1862 rows
Orders inserted: 5009 rows
Order Items inserted: 9994 rows


## Step 5 — Verify Data Loaded Successfully

In [5]:
# Verify row counts
verify_query = '''
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers UNION ALL
SELECT 'products', COUNT(*) FROM products UNION ALL
SELECT 'orders', COUNT(*) FROM orders UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
'''

result = pd.read_sql(verify_query, con=engine)
print('Table Row Counts:')
print(result.to_string(index=False))

Table Row Counts:
 table_name  row_count
  customers        793
   products       1862
     orders       5009
order_items       9994


## Step 6 — Basic Data Exploration

In [6]:
# Dataset overview
print('Dataset Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()

Dataset Shape: (9994, 21)

Data Types:
Row_ID             int64
Order_ID             str
Order_Date           str
Ship_Date            str
Ship_Mode            str
Customer_ID          str
Customer_Name        str
Segment              str
Country              str
City                 str
State                str
Postal_Code        int64
Region               str
Product_ID           str
Category             str
Sub_Category         str
Product_Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Null Values:
Row_ID           0
Order_ID         0
Order_Date       0
Ship_Date        0
Ship_Mode        0
Customer_ID      0
Customer_Name    0
Segment          0
Country          0
City             0
State            0
Postal_Code      0
Region           0
Product_ID       0
Category         0
Sub_Category     0
Product_Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: in

,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [7]:
# Unique categories
print('Unique Categories:', df['Category'].unique())
print('Unique Regions:', df['Region'].unique())
print('Unique Segments:', df['Segment'].unique())
print('Date Range:', df['Order_Date'].min(), 'to', df['Order_Date'].max())

Unique Categories: <ArrowStringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str
Unique Regions: <ArrowStringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str
Unique Segments: <ArrowStringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str
Date Range: 01-01-2017 to 9/30/2017


## Summary

| Table | Rows |
|-------|------|
| customers | 793 |
| products | 1,862 |
| orders | 5,009 |
| order_items | 9,994 |

Data successfully loaded into SQL Server. All SQL queries and analysis are in 'Superstore_Dataset_Week_2.sql'
